# 🗄️ SQL desde Cero — Guía para Ingenieros Industriales
**Nivel:** Principiante (no se requieren conocimientos previos de bases de datos)  
**Objetivo:** Dominar SQL para extraer, filtrar y analizar datos almacenados en bases de datos relacionales.

---

## ¿Por qué SQL para un Ingeniero Industrial?

En las empresas modernas, la mayoría de los datos operacionales viven en **bases de datos relacionales**:
- **ERP** (SAP, Oracle): órdenes de producción, inventarios, costos
- **MES** (Manufacturing Execution System): datos de máquinas, tiempos de ciclo
- **CMMS** (Mantenimiento): órdenes de trabajo, historial de fallas
- **WMS** (Almacén): movimientos de materiales, tiempos de picking

Sin SQL, dependes de que alguien más te "exporte el reporte". **Con SQL, accedes a los datos tú mismo.**

---

## ¿Qué es SQL?

**SQL** (Structured Query Language) es el lenguaje estándar para:
- **Consultar** datos (SELECT)
- **Filtrar** datos (WHERE)
- **Agregar** datos (GROUP BY, SUM, AVG...)
- **Combinar** tablas (JOIN)
- **Actualizar y crear** datos (INSERT, UPDATE, CREATE)

> 💡 En este notebook usamos **SQLite** (base de datos en Python, sin instalación extra).
> La sintaxis es idéntica a MySQL, PostgreSQL y SQL Server — aprende uno y sabes todos.

## 🏭 Nuestra base de datos: Planta de Manufactura

Vamos a crear una base de datos de una planta que fabrica componentes mecánicos.  
Tendrá 4 tablas relacionadas:

```
┌─────────────┐     ┌──────────────────┐     ┌──────────────────┐
│  maquinas   │────▶│ ordenes_produccion│────▶│ inspecciones     │
│─────────────│     │──────────────────│     │──────────────────│
│ id_maquina  │     │ id_orden         │     │ id_inspeccion    │
│ nombre      │     │ id_maquina (FK)  │     │ id_orden (FK)    │
│ tipo        │     │ id_producto (FK) │     │ resultado        │
│ año_instalac│     │ cantidad_objetivo │     │ defectos         │
└─────────────┘     │ cantidad_real    │     └──────────────────┘
                    │ fecha            │
┌─────────────┐     │ operador         │
│  productos  │────▶│ estado           │
│─────────────│     └──────────────────┘
│ id_producto │
│ nombre      │
│ categoria   │
│ precio_unit │
└─────────────┘
```

In [ ]:
import sqlite3
import pandas as pd

# Creamos la base de datos en memoria (para este notebook)
# En producción real usarías: conn = sqlite3.connect('mi_empresa.db')
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

print("✅ Conexión a SQLite establecida")
print("💡 Usaremos pd.read_sql() para mostrar resultados como tablas bonitas")

---
## 1️⃣ CREATE TABLE — Creando la estructura de la base de datos

Antes de guardar datos, definimos la **estructura** de cada tabla:  
qué columnas tiene, qué tipo de dato almacena cada una, y las relaciones entre tablas.

In [ ]:
# --- Crear tablas ---

cursor.executescript("""
-- Tabla de máquinas
CREATE TABLE maquinas (
    id_maquina     INTEGER PRIMARY KEY,
    nombre         TEXT    NOT NULL,
    tipo           TEXT    NOT NULL,
    año_instalacion INTEGER,
    ubicacion      TEXT
);

-- Tabla de productos
CREATE TABLE productos (
    id_producto  INTEGER PRIMARY KEY,
    nombre       TEXT    NOT NULL,
    categoria    TEXT    NOT NULL,
    precio_unitario REAL NOT NULL
);

-- Tabla de órdenes de producción (relaciona máquinas y productos)
CREATE TABLE ordenes_produccion (
    id_orden          INTEGER PRIMARY KEY,
    id_maquina        INTEGER NOT NULL,
    id_producto       INTEGER NOT NULL,
    fecha             TEXT    NOT NULL,
    turno             TEXT    NOT NULL,
    operador          TEXT    NOT NULL,
    cantidad_objetivo INTEGER NOT NULL,
    cantidad_real     INTEGER,
    tiempo_paro_min   REAL    DEFAULT 0,
    estado            TEXT    DEFAULT 'Completada',
    FOREIGN KEY (id_maquina)  REFERENCES maquinas(id_maquina),
    FOREIGN KEY (id_producto) REFERENCES productos(id_producto)
);

-- Tabla de inspecciones de calidad
CREATE TABLE inspecciones (
    id_inspeccion INTEGER PRIMARY KEY,
    id_orden      INTEGER NOT NULL,
    resultado     TEXT    NOT NULL,
    defectos      INTEGER DEFAULT 0,
    tipo_defecto  TEXT,
    inspector     TEXT,
    FOREIGN KEY (id_orden) REFERENCES ordenes_produccion(id_orden)
);
""")

conn.commit()
print("✅ Tablas creadas exitosamente")
print("   - maquinas")
print("   - productos")
print("   - ordenes_produccion")
print("   - inspecciones")

---
## 2️⃣ INSERT INTO — Insertando datos reales

In [ ]:
# --- Insertar datos de máquinas ---
cursor.executemany("""
    INSERT INTO maquinas VALUES (?, ?, ?, ?, ?)
""", [
    (1, 'Torno CNC-01',     'Torno CNC',   2018, 'Nave A'),
    (2, 'Torno CNC-02',     'Torno CNC',   2019, 'Nave A'),
    (3, 'Fresadora FR-01',  'Fresadora',   2017, 'Nave B'),
    (4, 'Fresadora FR-02',  'Fresadora',   2021, 'Nave B'),
    (5, 'Prensa H-01',      'Prensa',      2015, 'Nave C'),
    (6, 'Prensa H-02',      'Prensa',      2022, 'Nave C'),
])

# --- Insertar productos ---
cursor.executemany("""
    INSERT INTO productos VALUES (?, ?, ?, ?)
""", [
    (1, 'Eje Ø25mm',        'Ejes',      45.50),
    (2, 'Eje Ø50mm',        'Ejes',      82.00),
    (3, 'Brida Tipo A',     'Bridas',    28.75),
    (4, 'Brida Tipo B',     'Bridas',    35.20),
    (5, 'Carcasa Motor S1', 'Carcasas', 120.00),
    (6, 'Carcasa Motor S2', 'Carcasas', 145.00),
])

conn.commit()
print("✅ Datos de máquinas y productos insertados")

In [ ]:
import random
random.seed(99)

# Generamos 120 órdenes de producción (datos simulados realistas)
operadores = ['García, J.', 'López, M.', 'Martínez, R.', 'Torres, A.', 'Ruiz, C.']
turnos = ['Mañana', 'Tarde', 'Noche']
ordenes = []
inspecciones = []

from datetime import date, timedelta
fecha_inicio = date(2024, 1, 1)

for i in range(1, 121):
    fecha = (fecha_inicio + timedelta(days=(i-1) // 3)).isoformat()
    maquina = random.randint(1, 6)
    producto = random.randint(1, 6)
    turno    = turnos[(i-1) % 3]
    operador = random.choice(operadores)
    objetivo = random.randint(100, 200)
    paro     = round(random.expovariate(1/10), 1)
    eficiencia = random.uniform(0.85, 0.99)
    real     = int(objetivo * eficiencia)
    estado   = 'Completada' if real >= objetivo * 0.9 else 'Incompleta'

    ordenes.append((i, maquina, producto, fecha, turno, operador,
                    objetivo, real, paro, estado))

    # Inspección para cada orden
    defectos = random.randint(0, max(1, int((1-eficiencia)*real*2)))
    resultado = 'Aprobada' if defectos / real < 0.04 else 'Rechazada'
    tipo = random.choice(['Dimensional', 'Superficial', 'Ninguno']) if defectos > 0 else 'Ninguno'
    inspector = random.choice(['Sánchez, P.', 'Gómez, L.', 'Jiménez, F.'])
    inspecciones.append((i, i, resultado, defectos, tipo, inspector))

cursor.executemany("""
    INSERT INTO ordenes_produccion VALUES (?,?,?,?,?,?,?,?,?,?)
""", ordenes)

cursor.executemany("""
    INSERT INTO inspecciones VALUES (?,?,?,?,?,?)
""", inspecciones)

conn.commit()
print(f"✅ {len(ordenes)} órdenes de producción insertadas")
print(f"✅ {len(inspecciones)} inspecciones de calidad insertadas")

---
## 3️⃣ SELECT — Consultando datos

Esta es la instrucción **más usada en SQL**. El 80% de tu trabajo será escribir SELECTs.

```sql
SELECT columnas
FROM   tabla
WHERE  condición
ORDER BY columna ASC/DESC
LIMIT  n_filas
```

In [ ]:
# --- SELECT básico: ver todas las máquinas ---
query = """
SELECT *
FROM   maquinas
ORDER BY nombre
"""
pd.read_sql(query, conn)

In [ ]:
# --- SELECT con columnas específicas ---
query = """
SELECT nombre, categoria, precio_unitario
FROM   productos
WHERE  categoria = 'Ejes'
ORDER BY precio_unitario DESC
"""
print("Productos de la categoría 'Ejes':")
pd.read_sql(query, conn)

In [ ]:
# --- SELECT con cálculos en columnas ---
query = """
SELECT 
    id_orden,
    fecha,
    turno,
    operador,
    cantidad_objetivo,
    cantidad_real,
    ROUND(cantidad_real * 100.0 / cantidad_objetivo, 1) AS eficiencia_pct,
    tiempo_paro_min,
    estado
FROM ordenes_produccion
WHERE estado = 'Incompleta'
ORDER BY fecha DESC
LIMIT 10
"""
print("Últimas 10 órdenes incompletas:")
pd.read_sql(query, conn)

---
## 4️⃣ WHERE — Filtrado avanzado

```sql
WHERE condición1 AND condición2    -- ambas deben cumplirse
WHERE condición1 OR  condición2    -- al menos una debe cumplirse
WHERE columna IN ('val1','val2')   -- valor en una lista
WHERE columna BETWEEN min AND max  -- rango de valores
WHERE columna LIKE 'patrón%'       -- búsqueda por texto
WHERE columna IS NULL              -- valores nulos
```

In [ ]:
# --- Órdenes del turno de noche con paro mayor a 20 minutos ---
query = """
SELECT id_orden, fecha, turno, operador, 
       cantidad_objetivo, cantidad_real, tiempo_paro_min
FROM   ordenes_produccion
WHERE  turno = 'Noche'
  AND  tiempo_paro_min > 20
ORDER BY tiempo_paro_min DESC
"""
print("Órdenes nocturnas con paro > 20 min:")
pd.read_sql(query, conn)

In [ ]:
# --- Órdenes de máquinas de la Nave A (tornos CNC) ---
query = """
SELECT id_orden, fecha, id_maquina, cantidad_real, estado
FROM   ordenes_produccion
WHERE  id_maquina IN (1, 2)
  AND  fecha BETWEEN '2024-01-01' AND '2024-01-15'
ORDER BY fecha, id_maquina
LIMIT 12
"""
print("Órdenes de tornos CNC — primera quincena de enero:")
pd.read_sql(query, conn)

---
## 5️⃣ GROUP BY + Funciones de Agregación

Este es el equivalente SQL de las **tablas dinámicas** de Excel y el `groupby` de Pandas.

| Función | Qué hace |
|---------|---------|
| `COUNT(*)` | Cuenta filas |
| `SUM(col)` | Suma valores |
| `AVG(col)` | Promedio |
| `MIN(col)` / `MAX(col)` | Mínimo / Máximo |
| `ROUND(val, n)` | Redondea a n decimales |

In [ ]:
# --- Producción total y eficiencia por turno ---
query = """
SELECT 
    turno,
    COUNT(*)                                              AS total_ordenes,
    SUM(cantidad_objetivo)                                AS objetivo_total,
    SUM(cantidad_real)                                    AS produccion_real,
    ROUND(AVG(tiempo_paro_min), 1)                        AS paro_promedio_min,
    ROUND(SUM(cantidad_real)*100.0/SUM(cantidad_objetivo),1) AS eficiencia_global_pct
FROM ordenes_produccion
GROUP BY turno
ORDER BY eficiencia_global_pct DESC
"""
print("=== RESUMEN DE PRODUCCIÓN POR TURNO ===")
pd.read_sql(query, conn)

In [ ]:
# --- Calidad por máquina (usando HAVING para filtrar grupos) ---
query = """
SELECT 
    op.id_maquina,
    m.nombre                                         AS maquina,
    COUNT(i.id_inspeccion)                           AS total_inspecciones,
    SUM(i.defectos)                                  AS defectos_totales,
    ROUND(SUM(i.defectos)*100.0/SUM(op.cantidad_real),2) AS tasa_defectos_pct,
    SUM(CASE WHEN i.resultado='Rechazada' THEN 1 ELSE 0 END) AS ordenes_rechazadas
FROM   inspecciones i
JOIN   ordenes_produccion op ON i.id_orden   = op.id_orden
JOIN   maquinas           m  ON op.id_maquina = m.id_maquina
GROUP BY op.id_maquina, m.nombre
HAVING tasa_defectos_pct > 0     -- solo máquinas con algún defecto
ORDER BY tasa_defectos_pct DESC
"""
print("=== ANÁLISIS DE CALIDAD POR MÁQUINA ===")
pd.read_sql(query, conn)

---
## 6️⃣ JOIN — Combinando tablas

Los **JOINs** son la característica más poderosa de SQL. Permiten combinar información de múltiples tablas.

```
INNER JOIN → solo filas con coincidencia en AMBAS tablas
LEFT JOIN  → todas las filas de la tabla izquierda + coincidencias de la derecha
```

```
  Tabla A    INNER JOIN    Tabla B         LEFT JOIN
  ┌───┐                  ┌───┐          ┌───┐
  │ A │  ───────────── →  │A∩B│    →    │ A │ + A∩B
  └───┘                  └───┘          └───┘
```

In [ ]:
# --- JOIN de 3 tablas: reporte completo de producción ---
query = """
SELECT 
    op.id_orden,
    op.fecha,
    op.turno,
    m.nombre                                              AS maquina,
    m.ubicacion                                           AS nave,
    p.nombre                                              AS producto,
    p.categoria,
    op.operador,
    op.cantidad_objetivo,
    op.cantidad_real,
    ROUND(op.cantidad_real*100.0/op.cantidad_objetivo, 1) AS eficiencia_pct,
    ROUND(op.cantidad_real * p.precio_unitario, 2)        AS valor_producido_usd,
    i.resultado                                           AS calidad,
    i.defectos
FROM      ordenes_produccion op
JOIN      maquinas   m ON op.id_maquina  = m.id_maquina
JOIN      productos  p ON op.id_producto = p.id_producto
JOIN      inspecciones i ON i.id_orden   = op.id_orden
ORDER BY  op.fecha DESC, op.id_orden DESC
LIMIT 10
"""
print("=== REPORTE INTEGRADO DE PRODUCCIÓN (últimas 10 órdenes) ===")
pd.read_sql(query, conn)

In [ ]:
# --- Valor monetario total producido por categoría de producto ---
query = """
SELECT 
    p.categoria,
    COUNT(op.id_orden)                                        AS ordenes,
    SUM(op.cantidad_real)                                     AS unidades_producidas,
    ROUND(SUM(op.cantidad_real * p.precio_unitario), 2)       AS valor_total_usd,
    ROUND(AVG(op.cantidad_real * p.precio_unitario), 2)       AS valor_promedio_orden,
    SUM(i.defectos)                                           AS defectos_totales
FROM      ordenes_produccion op
JOIN      productos   p ON op.id_producto = p.id_producto
JOIN      inspecciones i ON i.id_orden    = op.id_orden
GROUP BY  p.categoria
ORDER BY  valor_total_usd DESC
"""
print("=== VALOR DE PRODUCCIÓN POR CATEGORÍA ===")
pd.read_sql(query, conn)

---
## 7️⃣ Subconsultas y CTEs — SQL Avanzado

Las **subconsultas** permiten anidar queries dentro de queries.  
Los **CTEs** (Common Table Expressions, con `WITH`) hacen el código más legible.

In [ ]:
# --- Subconsulta: órdenes con eficiencia MENOR al promedio global ---
query = """
SELECT 
    id_orden, fecha, turno, operador,
    ROUND(cantidad_real*100.0/cantidad_objetivo, 1) AS eficiencia_pct
FROM ordenes_produccion
WHERE (cantidad_real*100.0/cantidad_objetivo) < (
    SELECT AVG(cantidad_real*100.0/cantidad_objetivo)
    FROM ordenes_produccion
)
ORDER BY eficiencia_pct ASC
LIMIT 10
"""
print("Órdenes con eficiencia POR DEBAJO del promedio global:")
df_sub = pd.read_sql(query, conn)
print(df_sub)
print(f"\n📊 Estas {len(df_sub)} son las peores del set de 10 mostradas.")

In [ ]:
# --- CTE: ranking de operadores por eficiencia ---
query = """
WITH eficiencia_operador AS (
    SELECT 
        operador,
        COUNT(*)                                              AS ordenes,
        ROUND(AVG(cantidad_real*100.0/cantidad_objetivo), 1) AS eficiencia_prom_pct,
        ROUND(AVG(tiempo_paro_min), 1)                       AS paro_prom_min,
        SUM(CASE WHEN estado='Incompleta' THEN 1 ELSE 0 END) AS ordenes_incompletas
    FROM ordenes_produccion
    GROUP BY operador
)
SELECT 
    operador,
    ordenes,
    eficiencia_prom_pct,
    paro_prom_min,
    ordenes_incompletas,
    RANK() OVER (ORDER BY eficiencia_prom_pct DESC) AS ranking
FROM eficiencia_operador
ORDER BY ranking
"""
print("=== RANKING DE OPERADORES POR EFICIENCIA ===")
pd.read_sql(query, conn)

---
## 8️⃣ Python + SQL juntos: el flujo de trabajo profesional

Así es como trabajarás en el mundo real: SQL extrae los datos, Python los analiza.

In [ ]:
# Extrae datos con SQL → analiza con Pandas → visualiza con Matplotlib

query = """
SELECT 
    op.fecha,
    m.nombre                                                AS maquina,
    SUM(op.cantidad_real)                                   AS produccion,
    ROUND(SUM(op.cantidad_real)*100.0/SUM(op.cantidad_objetivo),1) AS eficiencia_pct,
    SUM(i.defectos)                                         AS defectos,
    ROUND(SUM(i.defectos)*100.0/SUM(op.cantidad_real), 2)  AS tasa_defectos_pct
FROM   ordenes_produccion op
JOIN   maquinas      m ON op.id_maquina  = m.id_maquina
JOIN   inspecciones  i ON i.id_orden     = op.id_orden
GROUP BY op.fecha, m.nombre
ORDER BY op.fecha, m.nombre
"""

df_analisis = pd.read_sql(query, conn, parse_dates=['fecha'])
print(f"Datos extraídos: {df_analisis.shape[0]} filas × {df_analisis.shape[1]} columnas")
df_analisis.head()

In [ ]:
# --- Visualización final: dashboard de producción ---
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(2, 1, figsize=(13, 10))

# Panel superior: eficiencia por máquina a lo largo del tiempo
for maquina, grupo in df_analisis.groupby('maquina'):
    diario = grupo.set_index('fecha')['eficiencia_pct']
    axes[0].plot(diario.index, diario.values,
                 marker='o', markersize=3, linewidth=1.5, label=maquina)

axes[0].axhline(y=90, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Meta 90%')
axes[0].set_title('Eficiencia de Producción por Máquina', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Eficiencia (%)')
axes[0].legend(loc='lower left', fontsize=8, ncol=3)
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
axes[0].tick_params(axis='x', rotation=30)

# Panel inferior: tasa de defectos diaria (todas las máquinas)
tasa_diaria = df_analisis.groupby('fecha')['tasa_defectos_pct'].mean()
axes[1].fill_between(tasa_diaria.index, tasa_diaria.values,
                     alpha=0.4, color='tomato')
axes[1].plot(tasa_diaria.index, tasa_diaria.values,
             color='firebrick', linewidth=2, marker='s', markersize=4)
axes[1].axhline(y=3.0, color='darkorange', linestyle='--',
                linewidth=2, label='Alerta: 3%')
axes[1].set_title('Tasa de Defectos Diaria — Promedio Planta', fontweight='bold', fontsize=13)
axes[1].set_ylabel('Tasa de Defectos (%)')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout(pad=3)
plt.savefig('sql_dashboard_produccion.png', dpi=120, bbox_inches='tight')
plt.show()
print("💾 Dashboard guardado como 'sql_dashboard_produccion.png'")

---
## 🏋️ Ejercicios SQL

### Ejercicio 1 — SELECT básico
Escribe una query que muestre todos los productos con precio mayor a $50, ordenados de mayor a menor precio.

### Ejercicio 2 — GROUP BY
¿Cuántas órdenes completó cada operador? Muestra el total de órdenes y el promedio de piezas producidas por orden.

### Ejercicio 3 — JOIN
Crea un reporte que muestre: nombre de la máquina, tipo de máquina, total de órdenes producidas, y tasa de aprobación de inspecciones (% de órdenes con resultado 'Aprobada').

### Ejercicio 4 — Filtro avanzado
Identifica qué tipo de defecto (`tipo_defecto`) es el más frecuente, excluyendo los registros donde `tipo_defecto = 'Ninguno'`.

### Ejercicio 5 — CTE avanzado
Escribe una consulta con CTE que calcule, por cada máquina, cuánto valor monetario se "perdió" a causa de defectos (considera: `defectos × precio_unitario` del producto correspondiente).

In [ ]:
# === SOLUCIÓN EJERCICIO 1 ===
query = """
SELECT nombre, categoria, precio_unitario
FROM   productos
WHERE  precio_unitario > 50
ORDER BY precio_unitario DESC
"""
print("Ejercicio 1 — Productos con precio > $50:")
pd.read_sql(query, conn)

In [ ]:
# === SOLUCIÓN EJERCICIO 4 — Tipo de defecto más frecuente ===
query = """
SELECT 
    tipo_defecto,
    COUNT(*) AS frecuencia,
    SUM(defectos) AS total_piezas_defectuosas
FROM inspecciones
WHERE tipo_defecto != 'Ninguno'
GROUP BY tipo_defecto
ORDER BY frecuencia DESC
"""
print("Ejercicio 4 — Tipos de defecto más frecuentes:")
pd.read_sql(query, conn)

In [ ]:
# Cerramos la conexión al finalizar
conn.close()
print("✅ Conexión cerrada correctamente")
print()
print("📚 Resumen de lo aprendido:")
print("   ✔ CREATE TABLE — definir estructura de base de datos")
print("   ✔ INSERT INTO  — cargar datos")
print("   ✔ SELECT       — consultar datos")
print("   ✔ WHERE        — filtrar registros")
print("   ✔ GROUP BY     — agrupar y agregar")
print("   ✔ JOIN         — combinar tablas")
print("   ✔ CTE (WITH)   — queries complejas legibles")
print("   ✔ Python + SQL — flujo de trabajo profesional")